# 01 -- Data Exploration with PySpark

Explore the Project Gutenberg sample dataset using PySpark in local mode.

**Goals:**
- Load book .txt files into Spark
- Compute basic statistics (count, word count, file size)
- Demo text cleaning pipeline
- Verify data readiness for LSH

In [ ]:
from config.settings import config
from scripts import clean_gutenberg_text

print(f"Environment: {config.ENV}")
print(f"Spark master: {config.SPARK_MASTER}")
print(f"Data path: {config.DATA_RAW_PATH}")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master(config.SPARK_MASTER)
    .appName(config.SPARK_APP_NAME)
    .config("spark.driver.memory", config.SPARK_DRIVER_MEMORY)
    .config("spark.executor.memory", config.SPARK_EXECUTOR_MEMORY)
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"Master: {sc.master}")
print(f"Parallelism: {sc.defaultParallelism}")

## 1. Load Book Files
Read all .txt files from `data/sample/` into a Spark DataFrame.

In [ ]:
import os

sample_dir = config.DATA_RAW_PATH
files = [f for f in os.listdir(sample_dir) if f.endswith(".txt")]
print(f"Found {len(files)} book files")

# Read all text files using wholeTextFiles (returns (path, content) pairs)
rdd = sc.wholeTextFiles(f"file://{os.path.abspath(sample_dir)}/*.txt")
print(f"RDD partitions: {rdd.getNumPartitions()}")
print(f"Total books loaded: {rdd.count()}")

In [ ]:
from pyspark.sql import Row

def parse_book(pair):
    path, content = pair
    filename = os.path.basename(path)
    book_id = filename.replace("pg", "").replace(".txt", "")
    lines = content.split("\n")
    words = content.split()
    return Row(
        book_id=book_id,
        filename=filename,
        num_lines=len(lines),
        num_words=len(words),
        num_chars=len(content),
    )

books_df = rdd.map(parse_book).toDF()
books_df.cache()
books_df.show(10, truncate=False)

## 2. Basic Statistics

In [ ]:
from pyspark.sql import functions as F

stats = books_df.agg(
    F.count("book_id").alias("total_books"),
    F.mean("num_words").cast("int").alias("avg_words"),
    F.min("num_words").alias("min_words"),
    F.max("num_words").alias("max_words"),
    F.mean("num_chars").cast("int").alias("avg_chars"),
    F.sum("num_words").alias("total_words"),
)
stats.show(truncate=False)

In [ ]:
import matplotlib.pyplot as plt

pdf = books_df.select("book_id", "num_words").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pdf["num_words"], bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("Word Count")
ax.set_ylabel("Number of Books")
ax.set_title("Word Count Distribution Across Sample Books")
ax.axvline(pdf["num_words"].mean(), color="red", linestyle="--", label=f"Mean: {pdf['num_words'].mean():.0f}")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Text Cleaning Demo
Use `scripts.clean_gutenberg_text()` to strip Gutenberg headers/footers and normalize text.

In [ ]:
# Pick one book to demonstrate cleaning
sample_path, sample_content = rdd.first()
sample_name = os.path.basename(sample_path)

print(f"Book: {sample_name}")
print(f"Raw length: {len(sample_content)} chars")
print(f"\n--- RAW (first 300 chars) ---")
print(sample_content[:300])

cleaned = clean_gutenberg_text(sample_content)
print(f"\n--- CLEANED (first 300 chars) ---")
print(cleaned[:300])
print(f"\nCleaned length: {len(cleaned)} chars")
print(f"Reduction: {(1 - len(cleaned)/len(sample_content))*100:.1f}%")

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

# Register UDF for cleaning
clean_udf = udf(clean_gutenberg_text, StringType())
word_count_udf = udf(lambda text: len(text.split()) if text else 0, IntegerType())

# Reload with content column
books_with_text = rdd.map(
    lambda pair: Row(
        book_id=os.path.basename(pair[0]).replace("pg", "").replace(".txt", ""),
        raw_text=pair[1],
    )
).toDF()

cleaned_df = (
    books_with_text
    .withColumn("cleaned_text", clean_udf("raw_text"))
    .withColumn("raw_words", word_count_udf("raw_text"))
    .withColumn("cleaned_words", word_count_udf("cleaned_text"))
    .select("book_id", "raw_words", "cleaned_words")
)

cleaned_df.cache()
cleaned_df.show(10)

# Summary
cleaned_df.agg(
    F.mean("raw_words").cast("int").alias("avg_raw_words"),
    F.mean("cleaned_words").cast("int").alias("avg_cleaned_words"),
).show()

## 4. Data Readiness for LSH Pipeline

The data is ready for the LSH pipeline:
- Books loaded into Spark DataFrames
- Text cleaning removes Gutenberg boilerplate
- Each book has a unique ID (from filename)
- Cleaned text ready for: shingling -> minhash -> LSH bucketing

**Next steps:** `02_preprocessing_demo.ipynb`

In [ ]:
# Show top 10 books by word count after cleaning
sample_cleaned = cleaned_df.join(
    books_with_text, on="book_id"
).select("book_id", "cleaned_words").orderBy(F.desc("cleaned_words"))

print("Top 10 books by word count (after cleaning):")
sample_cleaned.show(10, truncate=False)

In [ ]:
spark.stop()
print("SparkSession stopped. Data exploration complete.")